# 03 Machine Learning Preprocessing

This notebook builds the model-ready feature matrix using the same reusable preprocessing logic that powers the production scoring pipeline.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
import pandas as pd

from src.preprocessing import (
    TARGET_COLUMN,
    fit_preprocessing_artifacts,
    split_clean_dataset,
    transform_features,
)

clean_df = pd.read_csv(DATA_DIR / 'processed' / 'clean_telco_churn.csv')
clean_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
clean_df['ChurnFlag'] = clean_df[TARGET_COLUMN].map({'No': 0, 'Yes': 1})
clean_df[['Churn', 'ChurnFlag']].head()

,Churn,ChurnFlag
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


In [3]:
X_train_raw, X_test_raw, y_train, y_test = split_clean_dataset(clean_df, test_size=0.2, random_state=42)
print('X_train_raw shape:', X_train_raw.shape)
print('X_test_raw shape:', X_test_raw.shape)
print('Train churn rate:', y_train.mean().round(3))
print('Test churn rate:', y_test.mean().round(3))

X_train_raw shape: (5625, 20)
X_test_raw shape: (1407, 20)
Train churn rate: 0.266
Test churn rate: 0.266


In [4]:
categorical_columns = X_train_raw.select_dtypes(include='object').columns.tolist()
numerical_columns = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Categorical columns:', categorical_columns)
print('Numerical columns:', numerical_columns)

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'ChurnFlag']


In [5]:
preprocessing_artifacts = fit_preprocessing_artifacts(X_train_raw)
print('Number of encoded features:', len(preprocessing_artifacts['feature_columns']))
print('Default inference values:')
pd.Series(preprocessing_artifacts['default_input_values'])

Number of encoded features: 31
Default inference values:


gender                          Male
SeniorCitizen                    0.0
Partner                           No
Dependents                        No
tenure                          29.0
PhoneService                     Yes
MultipleLines                     No
InternetService          Fiber optic
OnlineSecurity                    No
OnlineBackup                      No
DeviceProtection                  No
TechSupport                       No
StreamingTV                       No
StreamingMovies                   No
Contract              Month-to-month
PaperlessBilling                 Yes
PaymentMethod       Electronic check
MonthlyCharges                  70.6
TotalCharges                 1410.25
ChurnFlag                        0.0
dtype: object

In [6]:
X_train = transform_features(X_train_raw, preprocessing_artifacts, strict=True)
X_test = transform_features(X_test_raw, preprocessing_artifacts, strict=True)
display(X_train.head())
print('Processed train shape:', X_train.shape)
print('Processed test shape:', X_test.shape)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,ChurnFlag,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
1408,-0.439319,1.321816,0.981556,1.659900,-0.601652,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6992,-0.439319,-0.267410,-0.971546,-0.562252,-0.601652,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3349,-0.439319,1.444064,0.837066,1.756104,-0.601652,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4486,-0.439319,-1.204646,0.641092,-0.908326,-0.601652,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3535,-0.439319,0.669826,-0.808787,-0.101561,-0.601652,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Processed train shape: (5625, 31)
Processed test shape: (1407, 31)


In [7]:
X_train.to_csv(DATA_DIR / 'processed' / 'X_train.csv', index=False)
X_test.to_csv(DATA_DIR / 'processed' / 'X_test.csv', index=False)
y_train.to_frame(name=TARGET_COLUMN).to_csv(DATA_DIR / 'processed' / 'y_train.csv', index=False)
y_test.to_frame(name=TARGET_COLUMN).to_csv(DATA_DIR / 'processed' / 'y_test.csv', index=False)
print('Processed datasets saved to data/processed/')

Processed datasets saved to data/processed/


## Pipeline Notes

- Categorical variables are encoded with `pd.get_dummies(..., drop_first=True)`.
- Numerical columns are standardized with `StandardScaler` fit on the training split only.
- The preprocessing metadata stores the schema, category levels, defaults, and scaler so training and inference stay aligned.